In [ ]:
"""
Script for Removal of HR and PR Windows Based on Outlier Detection and Peak Difference

This script removes specific time windows from HR (ECG) and PR (PPG) data based on:
- Outliers: HR/PR values outside (mean ± 3*std)
- Excessive HR-PR differences (|ECG HR - PPG HR|)

Main operations:
- Load previously saved `.npz` files containing preprocessed ECG and PPG signals and peaks.
- Align PR to HR (synchronization).
- Identify and remove bad windows based on defined thresholds.
- Plot signals before/after window removal.
- Plot signals, HR and PR and highlighted bad windows.
"""


# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import os
import numpy as np

from utils_remove_windows import (
    align_pr_to_hr, 
    compute_peaks_difference,
    compute_mean_std,
    extract_good_windows, 
    plot_hr_pr_before_after_removing,
    plot_hr_pr_and_bad_windows
)

%matplotlib qt


# =============================================================================
# DEFINE PARAMETERS
# =============================================================================

# Path of npz file
path = 'C:/Users/ilari/Desktop/Sleep disorders/Example files/Database Paper/Database Paper - Peaks/Pz 13.npz'

# Parameters (used for the selected method)
# - window_sec: duration (in seconds) of the analysis window (10)
# - th_n_peaks: maximum number of |HR - PR| peaks allowed in a window (1)
# - th_n_outliers: maximum number of HR/PR outliers allowed in a window (based on mean ± 3*std) (1)
# - h_min: minimum height threshold for |HR - PR| peak detection (20)

window_sec = 10
th_n_peaks = 1
th_n_outliers = 1
h_min = 20      


# =============================================================================
# LOAD DATA (.npz)
# =============================================================================

subj = os.path.splitext(os.path.basename(path))[0]
print(f'--- {subj} Analysis ---')

data = np.load(path)

sleep_ecg = data["ecg"]
ecg_peaks_final = data["ecg_peaks"]
fs_ecg = data["ecg_sampling_rate"]

sleep_ppg = data["ppg"]
ppg_peaks_final = data["ppg_peaks"]
fs_ppg = data["ppg_sampling_rate"]

# Time vectors 
time_ecg = np.arange(len(sleep_ecg)) / fs_ecg  
time_ppg = np.arange(len(sleep_ppg)) / fs_ppg 


# =============================================================================
# ALIGNMENT PULSE RATE TO HEART RATE 
# =============================================================================

# Using cross-correlation to estimate and correct temporal delay (lag)
hr_final, pr_final, time_hr, time_pr, time_shift = align_pr_to_hr(
    ecg_peaks_final, ppg_peaks_final, fs_ecg, fs_ppg, len(sleep_ecg)
)


# =============================================================================
# BAD WINDOW DETECTION PARAMETERS
# =============================================================================

# Compute |HR - PR| difference and detect peaks above a minimum threshold (h_min)
# These peaks indicate mismatched or noisy intervals between HR and PR signals
time_common, diff_hr_pr, diff_peaks = compute_peaks_difference(
    hr_final, pr_final, time_hr, time_pr, h_min
)

# Compute HR and PR upper/lower bounds (mean ± 3*std)
# Used to detect outliers in the HR and PR signals
hr_bounds, pr_bounds = compute_mean_std(hr_final, pr_final)


# =============================================================================
# WINDOW REMOVAL 
# =============================================================================

# Extract and reconstruct "clean" HR and PR segments based on outlier thresholds and |HR–PR| peaks count
hr_clean, pr_clean, time_hr_clean, time_pr_clean, windows_ranges, good_windows, n_peaks_per_window = extract_good_windows(
    hr_final, time_hr, pr_final, time_pr,
    diff_hr_pr, time_common, diff_peaks, 
    hr_bounds, pr_bounds, window_sec, th_n_outliers, th_n_peaks
)

# Plot HR and PR before/after windows removing
plot_hr_pr_before_after_removing(
    time_hr, hr_final, time_pr, pr_final,
    time_hr_clean, hr_clean, time_pr_clean, pr_clean,
    subj, window_sec, th_n_outliers, th_n_peaks
)

# Plot ECG and PPG signals, HR and PR values, diff_hr_pr with Peaks and highlight bad windows
plot_hr_pr_and_bad_windows(
    time_hr, hr_final, hr_bounds,
    time_pr, pr_final, pr_bounds, 
    good_windows, windows_ranges,
    time_ecg, sleep_ecg, ecg_peaks_final,
    time_ppg, sleep_ppg, ppg_peaks_final,
    diff_hr_pr, time_common, diff_peaks,
    n_peaks_per_window, subj, window_sec, th_n_outliers, th_n_peaks
)   


--- Pz 13 Analysis ---
Lag: 95 samples (0.371 s)
Good windows: 2471/2570 (96.15%)
